[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksankaran/hello-model/blob/main/mini_gpt.ipynb)

# Attention Is All You Need (To Understand)

We've gone from a straight line (2 knobs) to a neural network (10 knobs) to PyTorch (same knobs, free gradients).

Now we build the architecture behind every major language model: the **transformer**.

We'll train a character-level model on Shakespeare. It will learn to generate text that looks (roughly) like Shakespeare, starting from random noise. About 150 lines of model code and ~210,000 parameters.

GPT-4 uses the same architecture with 1.8 trillion parameters. Same building blocks, same training loop.

---
## Part 1: Get the Data

We'll use a small chunk of Shakespeare (~1MB of text). The task: given a sequence of characters, predict the next one.

In [ ]:
import urllib.request

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = urllib.request.urlopen(url).read().decode('utf-8')

print(f"Total characters: {len(text):,}")
print(f"\nFirst 200 characters:")
print(text[:200])

---
## Part 2: Character-Level Tokenization

We turn each character into a number. Our vocabulary is every unique character in the text.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Encode: character -> integer
# Decode: integer -> character
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda l: ''.join(idx_to_char[i] for i in l)

print(f"\nencode('hello') = {encode('hello')}")
print(f"decode([46, 43, 50, 50, 53]) = '{decode([46, 43, 50, 50, 53])}'")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Encode the entire text as a tensor of integers
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}")
print(f"First 20 tokens: {data[:20]}")
print(f"Which decodes to: '{decode(data[:20].tolist())}'")

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"\nTraining tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

---
## Part 3: Data Preparation

Each training example is a window of characters. The input is the window, the target is the same window shifted by one character.

```
Input:  "First Citizen:"     (characters 0-13)
Target: "irst Citizen:\n"    (characters 1-14)
```

At every position, the model predicts the next character.

In [ ]:
# Hyperparameters
block_size = 32     # context window (characters the model can look back at)
batch_size = 32     # how many sequences to process in parallel
embed_dim = 64      # size of each token's vector representation
num_heads = 4       # parallel attention patterns
num_layers = 4      # stacked transformer blocks
learning_rate = 1e-3
max_steps = 5000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def get_batch(split):
    """Get a random batch of input/target pairs."""
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([d[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

# Example batch
xb, yb = get_batch('train')
print(f"\nInput shape:  {xb.shape}  (batch_size x block_size)")
print(f"Target shape: {yb.shape}")
print(f"\nFirst sequence input:  '{decode(xb[0].tolist())}'")
print(f"First sequence target: '{decode(yb[0].tolist())}'")

---
## Part 4: Self-Attention

The key innovation of transformers. Each token looks at every previous token and decides: **"which of you should I pay attention to?"**

Each token produces three vectors:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?"
- **Value (V):** "What do I share if selected?"

The attention score between two tokens = how well the Query of one matches the Key of the other. Higher score = more attention = more of that token's Value gets mixed in.

**Causal masking:** A token can only attend to tokens before it (no peeking at the future).

In [ ]:
class AttentionHead(nn.Module):
    """Single head of self-attention."""

    def __init__(self, embed_dim, head_dim, block_size):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key   = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        # Causal mask: lower triangular matrix
        self.register_buffer('mask',
            torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)   # (B, T, head_dim)
        k = self.key(x)     # (B, T, head_dim)
        v = self.value(x)   # (B, T, head_dim)

        # Attention scores: how much should each token attend to each other?
        scores = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)

        # Causal mask: can't look at future tokens
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))

        # Softmax: convert to probabilities
        weights = F.softmax(scores, dim=-1)

        # Weighted sum of values
        return weights @ v

print("AttentionHead defined.")
print(f"For embed_dim={embed_dim}, num_heads={num_heads}: head_dim = {embed_dim // num_heads}")

### Multi-Head Attention

One head learns one kind of pattern. Multiple heads learn different patterns in parallel - one might track character pairs, another might track word boundaries, another might track sentence structure.

Run several heads, concatenate their outputs.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multiple attention heads in parallel."""

    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim, block_size)
             for _ in range(num_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # Run all heads in parallel, concatenate outputs
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

print(f"MultiHeadAttention: {num_heads} heads, each with dim {embed_dim // num_heads}")

---
## Part 5: The Transformer Block

A transformer block combines:
1. **Multi-head attention** - tokens communicate with each other
2. **Feed-forward network** - each token processes its information independently
3. **Residual connections** - shortcuts so gradients flow easily (like the domino analogy from Part 2)
4. **Layer normalization** - keeps numbers stable through many layers

In [ ]:
class TransformerBlock(nn.Module):
    """One transformer block: attention + feed-forward with residual connections."""

    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads, block_size)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))   # attend + skip connection
        x = x + self.ff(self.norm2(x))          # transform + skip connection
        return x

print("TransformerBlock defined.")

---
## Part 6: The Full Model - MiniGPT

Stack the pieces:
1. **Token embedding** - each character becomes a learnable vector
2. **Positional embedding** - each position gets its own vector (so the model knows order)
3. **Transformer blocks** - stacked attention + feed-forward layers
4. **Output head** - project back to vocabulary size for next-character prediction

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, block_size):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(block_size, embed_dim)
        self.blocks      = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads, block_size)
              for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)
        self.block_size = block_size

    def forward(self, x):
        B, T = x.shape
        tok = self.token_embed(x)                              # (B, T, embed_dim)
        pos = self.pos_embed(torch.arange(T, device=x.device)) # (T, embed_dim)
        x = tok + pos
        x = self.blocks(x)
        x = self.norm(x)
        return self.head(x)                                    # (B, T, vocab_size)

    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generate text one character at a time."""
        for _ in range(max_new_tokens):
            # Crop to block_size
            context = idx[:, -self.block_size:]
            logits = self(context)
            # Only look at the last position's prediction
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Create the model
torch.manual_seed(42)
model = MiniGPT(
    vocab_size  = vocab_size,
    embed_dim   = embed_dim,
    num_heads   = num_heads,
    num_layers  = num_layers,
    block_size  = block_size,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT created with {num_params:,} parameters")
print(f"\nFor comparison:")
print(f"  Our line model (Part 1):      2 parameters")
print(f"  Our neural network (Part 2):   10 parameters")
print(f"  MiniGPT (this notebook):       {num_params:,} parameters")
print(f"  GPT-4:                         ~1,800,000,000,000 parameters")
print(f"\nSame architecture. Different scale.")

### Before training: pure noise

With random weights, the model's predictions are random.

In [ ]:
# Generate text before training
start = torch.zeros((1, 1), dtype=torch.long, device=device)
pre_training = decode(model.generate(start, max_new_tokens=200)[0].tolist())
print("Generated text BEFORE training (random weights):")
print("=" * 50)
print(pre_training)
print("=" * 50)
print("\nPure noise. Every character is a random guess.")

---
## Part 7: Training

Same loop as always: predict, measure, adjust, repeat.

The loss function is **cross-entropy**: for each position, the model outputs a probability for each of the 65 characters. Cross-entropy measures how much probability it put on the correct one. Random guessing across 65 characters gives a loss of about 4.17 (-ln(1/65)).

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_steps=200):
    """Estimate loss on train and val sets."""
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        total = 0
        for _ in range(eval_steps):
            xb, yb = get_batch(split)
            logits = model(xb)
            loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))
            total += loss.item()
        losses[split] = total / eval_steps
    model.train()
    return losses

# Check initial loss (should be ~4.17 = random guessing)
initial = estimate_loss(model, eval_steps=50)
print(f"Initial loss: train={initial['train']:.2f}, val={initial['val']:.2f}")
print(f"Random guessing would give: {-torch.log(torch.tensor(1.0/vocab_size)).item():.2f}")

In [ ]:
# --- TRAINING LOOP ---
# Same structure as every previous notebook:
# predict -> measure loss -> compute gradients -> adjust knobs -> repeat

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training MiniGPT for {max_steps} steps...\n")

for step in range(max_steps):
    # Get a batch
    xb, yb = get_batch('train')

    # Forward pass: predict next character at each position
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    # Backward pass + update (same 3 lines as Part 3!)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress
    if step % 500 == 0 or step == max_steps - 1:
        losses = estimate_loss(model, eval_steps=100)
        print(f"Step {step:5d} | train loss: {losses['train']:.3f} | val loss: {losses['val']:.3f}")

print(f"\nTraining complete.")

---
## Part 8: Generate Text

Now let's see what the model learned. We give it a starting character and let it predict one character at a time.

In [ ]:
# Generate text after training
start = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = decode(model.generate(start, max_new_tokens=500)[0].tolist())

print("Generated text AFTER training:")
print("=" * 50)
print(generated)
print("=" * 50)

In [ ]:
# Try different temperatures
print("=" * 50)
print("TEMPERATURE = 0.5 (more conservative, repetitive)")
print("=" * 50)
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, max_new_tokens=300, temperature=0.5)[0].tolist()))

print()
print("=" * 50)
print("TEMPERATURE = 1.5 (more creative, chaotic)")
print("=" * 50)
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, max_new_tokens=300, temperature=1.5)[0].tolist()))

---
## Part 9: What Just Happened

With ~210,000 parameters and a few minutes of training, our model learned:

- Character names in ALL CAPS followed by a colon
- Line breaks after dialogue
- English words (mostly real ones)
- Punctuation patterns
- Rough sentence structure

It learned all of this just from predicting the next character, one at a time.

The same architecture — embeddings, attention heads, transformer blocks — powers ChatGPT, Claude, Gemini, and every other large language model. The difference is scale:

| | MiniGPT (ours) | GPT-4 |
|---|---|---|
| Parameters | ~210K | ~1.8 trillion |
| Training data | 1MB Shakespeare | Trillions of tokens |
| Context window | 32 characters | 128K+ tokens |
| Training time | Minutes | Months |
| Hardware | 1 GPU | Thousands of GPUs |

Same architecture. Same training loop. Different scale.

---
## The Full Journey

| Part | Model | Parameters | Task |
|------|-------|------------|------|
| 1 | y = mx + b | 2 | Predict house prices |
| 2 | Neural network | 10 | Fit curved data |
| 3 | Same, in PyTorch | 10 | Same, with autograd |
| 4 | Transformer | ~210,000 | Generate Shakespeare |

From a straight line to a language model. The training loop never changed:

1. **Predict** — push data through the model
2. **Measure** — compute the loss
3. **Adjust** — backward pass + optimizer step
4. **Repeat**

That's all of AI. The rest is scale.

---
## Try It Yourself

Experiments:

1. **Different text:** Replace the Shakespeare URL with your own text file. Try song lyrics, Python code, or a novel.

2. **More layers:** Change `num_layers` from 4 to 8. Does the text quality improve?

3. **Wider model:** Change `embed_dim` from 64 to 128 (and train longer). More capacity = potentially better text.

4. **Longer context:** Change `block_size` from 32 to 64 or 128. The model can look further back.

5. **Temperature sampling:** In `model.generate()`, try `temperature=0.3` (conservative) vs `temperature=2.0` (wild).

6. **Prompt the model:** Instead of starting from a blank token, start with a specific string:

In [ ]:
# Start with a prompt instead of a blank token
prompt = "ROMEO:"
prompt_tokens = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
generated = decode(model.generate(prompt_tokens, max_new_tokens=300)[0].tolist())

print(f"Prompt: '{prompt}'")
print(f"\nGenerated continuation:")
print(generated)